In [ ]:
# ============================================================
# BLOCK 6: FINAL FORECAST FOR 2026-2027
# Project: Forecasting Household Deposit Volume in Russia
# Author: Nadezhda Silkina
# Date: 2026
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries loaded")

# ============================================================
# 2. LOAD DATA
# ============================================================

url = 'https://raw.githubusercontent.com/HopeSilkina/deposits_forecast_project/main/data/processed_deposits_data.xlsx'
df = pd.read_excel(url, sheet_name='data')
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

print(f"✅ Data loaded. Records: {len(df)}")
print(f"Period: from {df.index.min()} to {df.index.max()}")

# ============================================================
# 3. RESTORING RIDGE MODEL (FULL SET OF 38 FEATURES FROM BLOCK 4)
# ============================================================

print("\n" + "="*60)
print("3. RESTORING RIDGE MODEL (38 FEATURES)")
print("="*60)

# Create a copy for feature engineering
df_model = df.copy()

# 3.1. DEPOS lags
df_model['DEPOS_log'] = np.log(df_model['DEPOS'])
for lag in [1, 3, 6, 12]:
    df_model[f'DEPOS_lag_{lag}'] = df_model['DEPOS'].shift(lag)

# 3.2. Seasonal dummy variables
df_model['Month'] = df_model.index.month
for month in range(2, 13):
    df_model[f'month_{month}'] = (df_model['Month'] == month).astype(int)

# 3.3. Structural variables
df_model['post_2022'] = (df_model.index >= '2023-01-01').astype(int)
df_model['covid'] = ((df_model.index >= '2020-03-01') & (df_model.index <= '2022-01-01')).astype(int)

# 3.4. CRED1 regime (DEPOS > 32000) — from block 4
threshold_depos = 32000
df_model['regime_cred1'] = (df_model['DEPOS'] > threshold_depos).astype(int)

# 3.5. WAGE anomalies (adaptive threshold -2σ)
model_wage = LinearRegression()
model_wage.fit(df_model[['WAGE']].values, df_model['DEPOS'].values)
df_model['residual_wage'] = df_model['DEPOS'] - model_wage.predict(df_model[['WAGE']].values)
threshold_anomaly = -2.0 * df_model['residual_wage'].std()
df_model['anomaly_wage'] = (df_model['residual_wage'] < threshold_anomaly).astype(int)

# 3.6. Macro factor lags
for col in ['WAGE', 'CPI', 'USDind']:
    for lag in [1, 3, 6]:
        df_model[f'{col}_lag_{lag}'] = df_model[col].shift(lag)

# 3.7. Interaction UNEM × DEP1
df_model['UNEM_DEP1'] = df_model['UNEM'] * df_model['DEP1']

# Final feature set (full set from block 4, 38 features)
feature_columns = [
    'WAGE', 'SERV', 'DEP1', 'CRED1', 'CPI', 'USDind', 'UNEM', 'IPI', 'IMP',
    'DEPOS_lag_1', 'DEPOS_lag_3', 'DEPOS_lag_6', 'DEPOS_lag_12',
    'month_2', 'month_3', 'month_4', 'month_5', 'month_6',
    'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12',
    'post_2022', 'covid', 'regime_cred1', 'anomaly_wage',
    'WAGE_lag_1', 'WAGE_lag_3', 'WAGE_lag_6',
    'CPI_lag_1', 'CPI_lag_3', 'CPI_lag_6',
    'USDind_lag_1', 'USDind_lag_3', 'USDind_lag_6',
    'UNEM_DEP1'
]

print(f"📊 Total features: {len(feature_columns)}")
print(f"📌 Model: Ridge regression (alpha=1.0), full feature set from block 4")

# Prepare data for training
X = df_model[feature_columns].dropna()
y = df_model.loc[X.index, 'DEPOS']

print(f"📊 Observations with complete data: {len(X)}")
print(f"📊 Period: {X.index[0].strftime('%Y-%m')} — {X.index[-1].strftime('%Y-%m')}")

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train model on ALL data (134 observations)
ridge_final = Ridge(alpha=1.0)
ridge_final.fit(X_scaled, y)

# Check quality on training sample
y_pred_train = ridge_final.predict(X_scaled)
r2_train = r2_score(y, y_pred_train)
rmse_train = np.sqrt(mean_squared_error(y, y_pred_train))
mae_train = mean_absolute_error(y, y_pred_train)

print(f"\n✅ Ridge model trained on all data (n={len(X)})")
print(f"   R²_train = {r2_train:.4f}")
print(f"   RMSE_train = {rmse_train:.2f} billion RUB")
print(f"   MAE_train = {mae_train:.2f} billion RUB")
print(f"\n📌 NOTE: Model is retrained on all 134 observations to")
print(f"   maximize information use. Structure is identical to")
print(f"   the validated model (R²_test = 0.9422 on test sample).")

# ============================================================
# 4. PREPARING SCENARIOS FOR MACRO FACTORS
# ============================================================

print("\n" + "="*60)
print("4. PREPARING SCENARIOS FOR 12 MONTHS (March 2026 — February 2027)")
print("="*60)

# Last actual date
last_date = df.index[-1]
print(f"\nLast actual date: {last_date.strftime('%Y-%m-%d')}")

# Future dates: MARCH 2026 — FEBRUARY 2027 (12 months)
future_dates = pd.date_range(start='2026-03-31', periods=12, freq='ME')
print(f"Forecast period: {future_dates[0].strftime('%Y-%m')} — {future_dates[-1].strftime('%Y-%m')}")

# Last known values (February 2026)
last_values = df.iloc[-1]
last_depos = last_values['DEPOS']

print(f"\nCurrent DEPOS level (February 2026): {last_depos:,.0f} billion RUB")

# Function to create scenario
def create_scenario(growth_params, scenario_name):
    """
    Creates a scenario for macro factors for 12 months.

    growth_params: dict with growth/change parameters for each factor
    """
    scenario = pd.DataFrame(index=future_dates)

    # WAGE: growth according to scenario
    last_wage = last_values['WAGE']
    scenario['WAGE'] = [last_wage * (1 + growth_params['WAGE']) ** (i+1) for i in range(12)]

    # SERV: growth
    last_serv = last_values['SERV']
    scenario['SERV'] = [last_serv * (1 + growth_params['SERV']) ** (i+1) for i in range(12)]

    # DEP1: rate level
    scenario['DEP1'] = last_values['DEP1'] * (1 + growth_params['DEP1'])

    # CRED1
    scenario['CRED1'] = last_values['CRED1'] * (1 + growth_params['CRED1'])

    # CPI: level
    scenario['CPI'] = last_values['CPI'] + growth_params['CPI']

    # USDind: volatility
    scenario['USDind'] = growth_params['USDind']

    # UNEM: level
    scenario['UNEM'] = last_values['UNEM'] * (1 + growth_params['UNEM'])

    # IPI: index
    scenario['IPI'] = last_values['IPI'] * (1 + growth_params['IPI'])

    # IMP: imports
    scenario['IMP'] = last_values['IMP'] * (1 + growth_params['IMP'])

    # Structural variables
    scenario['post_2022'] = 1
    scenario['covid'] = 0
    scenario['anomaly_wage'] = 0
    scenario['regime_cred1'] = 1  # DEPOS > 32000 — forecast above this threshold

    # Interaction
    scenario['UNEM_DEP1'] = scenario['UNEM'] * scenario['DEP1']

    # Initialize DEPOS and macro factor lags
    for col in ['DEPOS_lag_1', 'DEPOS_lag_3', 'DEPOS_lag_6', 'DEPOS_lag_12']:
        scenario[col] = np.nan

    for col in ['WAGE', 'CPI', 'USDind']:
        for lag in [1, 3, 6]:
            scenario[f'{col}_lag_{lag}'] = np.nan

    return scenario

# Define scenarios
scenarios = {
    'Baseline': {
        'WAGE': 0.007,    # +0.7% per month (~8.7% annually)
        'SERV': 0.005,    # +0.5% per month
        'DEP1': -0.02,    # -2% (rate decrease)
        'CRED1': -0.01,   # -1%
        'CPI': -0.02,     # -0.02 p.p. (inflation decreasing)
        'USDind': 0.3,    # moderate volatility
        'UNEM': 0.01,     # +1% (slight increase)
        'IPI': 0.002,     # +0.2% per month
        'IMP': 0.003      # +0.3% per month
    },
    'Optimistic': {
        'WAGE': 0.01,     # +1% per month (~12.7% annually)
        'SERV': 0.008,    # +0.8% per month
        'DEP1': -0.05,    # -5% (rate decrease)
        'CRED1': -0.03,   # -3%
        'CPI': -0.05,     # -0.05 p.p. (inflation rapidly decreasing)
        'USDind': 0.1,    # low volatility
        'UNEM': -0.02,    # -2% (unemployment decrease)
        'IPI': 0.005,     # +0.5% per month
        'IMP': 0.006      # +0.6% per month
    },
    'Pessimistic': {
        'WAGE': 0.004,    # +0.4% per month (~4.9% annually)
        'SERV': 0.002,    # +0.2% per month
        'DEP1': 0.05,     # +5% (rate increase)
        'CRED1': 0.03,    # +3%
        'CPI': 0.05,      # +0.05 p.p. (inflation rising)
        'USDind': 0.8,    # high volatility
        'UNEM': 0.05,     # +5% (unemployment increase)
        'IPI': -0.003,    # -0.3% per month
        'IMP': -0.005     # -0.5% per month
    }
}

print("\n📊 Macro factor scenarios:")
for name, params in scenarios.items():
    print(f"\n   {name}:")
    print(f"     WAGE: {params['WAGE']*100:+.1f}%/month")
    print(f"     CPI: {params['CPI']:+.2f} p.p.")
    print(f"     UNEM: {params['UNEM']*100:+.1f}%")
    print(f"     DEP1: {params['DEP1']*100:+.1f}%")

# ============================================================
# 5. FORECAST FOR 12 MONTHS
# ============================================================

print("\n" + "="*60)
print("5. FORECAST FOR 12 MONTHS (March 2026 — February 2027)")
print("="*60)

# Function for scenario forecasting
def forecast_scenario(growth_params, scenario_name):
    """
    Iterative forecast for 12 months.
    DEPOS lags are updated at each step.
    """
    scenario = create_scenario(growth_params, scenario_name)

    predictions = []
    current_depos = last_depos

    # DEPOS history for lags (actual values)
    dep_history = df['DEPOS'].tolist()

    for i, date in enumerate(future_dates):
        # DEPOS lags: use actual where possible, then forecast values
        scenario.loc[date, 'DEPOS_lag_1'] = dep_history[-1]
        scenario.loc[date, 'DEPOS_lag_3'] = dep_history[-3] if len(dep_history) >= 3 else dep_history[0]
        scenario.loc[date, 'DEPOS_lag_6'] = dep_history[-6] if len(dep_history) >= 6 else dep_history[0]
        scenario.loc[date, 'DEPOS_lag_12'] = dep_history[-12] if len(dep_history) >= 12 else dep_history[0]

        # Seasonal variables
        month_num = date.month
        for m in range(2, 13):
            scenario.loc[date, f'month_{m}'] = 1 if month_num == m else 0

        # Macro factor lags
        for col in ['WAGE', 'CPI', 'USDind']:
            # Lag 1: previous month (actual or forecast)
            if i == 0:
                scenario.loc[date, f'{col}_lag_1'] = df[col].iloc[-1]
            else:
                scenario.loc[date, f'{col}_lag_1'] = scenario.loc[future_dates[i-1], col]

            # Lag 3: 3 months ago
            if i < 3:
                scenario.loc[date, f'{col}_lag_3'] = df[col].iloc[-3+i]
            else:
                scenario.loc[date, f'{col}_lag_3'] = scenario.loc[future_dates[i-3], col]

            # Lag 6: 6 months ago
            if i < 6:
                scenario.loc[date, f'{col}_lag_6'] = df[col].iloc[-6+i]
            else:
                scenario.loc[date, f'{col}_lag_6'] = scenario.loc[future_dates[i-6], col]

        # Form feature vector
        X_pred = scenario.loc[date, feature_columns].values.reshape(1, -1)
        X_pred_scaled = scaler.transform(X_pred)

        # Forecast
        pred = ridge_final.predict(X_pred_scaled)[0]
        predictions.append(pred)

        # Update history
        dep_history.append(pred)

    scenario['DEPOS_forecast'] = predictions
    return scenario, predictions

# Forecasts for all scenarios
forecasts = {}
for name, params in scenarios.items():
    print(f"\n🔧 Forecasting scenario «{name}»...")
    scenario, predictions = forecast_scenario(params, name)
    forecasts[name] = {
        'scenario': scenario,
        'predictions': predictions
    }
    print(f"   Done: {len(predictions)} values")
    print(f"   Range: {min(predictions):,.0f} — {max(predictions):,.0f} billion RUB")

# ============================================================
# 6. CONFIDENCE INTERVALS
# ============================================================

print("\n" + "="*60)
print("6. CONFIDENCE INTERVALS")
print("="*60)

# Standard deviation of residuals on training sample
residuals_train = y - y_pred_train
std_residuals = residuals_train.std()

print(f"Standard deviation of residuals: {std_residuals:.2f} billion RUB")
print(f"95% confidence interval: ±{1.96 * std_residuals:.2f} billion RUB")

# Add intervals to baseline scenario
base_predictions = np.array(forecasts['Baseline']['predictions'])
lower_bound = base_predictions - 1.96 * std_residuals
upper_bound = base_predictions + 1.96 * std_residuals

# ============================================================
# 7. FORECAST VISUALIZATION
# ============================================================

print("\n" + "="*60)
print("7. FORECAST VISUALIZATION")
print("="*60)

plt.figure(figsize=(16, 8))

# Historical data (last 24 months for context)
history_start = df.index[-24]
plt.plot(df.loc[history_start:].index, df.loc[history_start:, 'DEPOS'],
         label='Actual data', color='#1f77b4', linewidth=2.5)

# Scenario forecasts
colors = {'Baseline': '#2ca02c', 'Optimistic': '#ff7f0e', 'Pessimistic': '#d62728'}

for name, forecast_data in forecasts.items():
    predictions = forecast_data['predictions']
    plt.plot(future_dates, predictions, label=f'Forecast: {name}',
             color=colors[name], linestyle='--', linewidth=2)

# Confidence interval for baseline scenario
plt.fill_between(future_dates, lower_bound, upper_bound,
                 alpha=0.2, color='#2ca02c', label='95% CI (baseline)')

# Vertical line — forecast start
plt.axvline(x=last_date, color='gray', linestyle=':', alpha=0.5, label='Forecast start')

plt.title('Household Deposit Volume Forecast\nMarch 2026 — February 2027 (3 scenarios)',
          fontsize=14)
plt.xlabel('Date')
plt.ylabel('Deposit volume, billion RUB')
plt.legend(loc='upper left', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('06_final_forecast_2026_2027.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Plot saved: 06_final_forecast_2026_2027.png")

# ============================================================
# 8. FINAL FORECAST TABLE
# ============================================================

print("\n" + "="*60)
print("8. FINAL FORECAST TABLE")
print("="*60)

# Create table
forecast_table = pd.DataFrame({
    'Month': [d.strftime('%Y-%m') for d in future_dates],
    'Baseline': forecasts['Baseline']['predictions'],
    'Optimistic': forecasts['Optimistic']['predictions'],
    'Pessimistic': forecasts['Pessimistic']['predictions'],
    'Lower bound 95%': lower_bound,
    'Upper bound 95%': upper_bound
})

forecast_table = forecast_table.round(0)
print("\n📊 Deposit volume forecast (billion RUB):")
print(forecast_table.to_string(index=False))

# Save to CSV
forecast_table.to_csv('06_forecast_2026_2027.csv', index=False)
print("\n✅ Table saved: 06_forecast_2026_2027.csv")

# ============================================================
# 9. SUMMARY STATISTICS
# ============================================================

print("\n" + "="*60)
print("9. SUMMARY STATISTICS OF FORECAST")
print("="*60)

print(f"\n📊 Current level (February 2026): {last_depos:,.0f} billion RUB")

for name in ['Baseline', 'Optimistic', 'Pessimistic']:
    pred = forecasts[name]['predictions']
    print(f"\n   {name} scenario:")
    print(f"     Min: {min(pred):,.0f} billion RUB ({future_dates[np.argmin(pred)].strftime('%B %Y')})")
    print(f"     Max: {max(pred):,.0f} billion RUB ({future_dates[np.argmax(pred)].strftime('%B %Y')})")
    print(f"     Average: {np.mean(pred):,.0f} billion RUB")
    print(f"     By February 2027: {pred[-1]:,.0f} billion RUB")
    print(f"     Growth over 12 months: {(pred[-1] - last_depos) / last_depos * 100:+.1f}%")

# ============================================================
# 10. FINAL SUMMARY
# ============================================================

print("\n" + "="*60)
print("10. FINAL SUMMARY")
print("="*60)

base_final = forecasts['Baseline']['predictions'][-1]
opt_final = forecasts['Optimistic']['predictions'][-1]
pes_final = forecasts['Pessimistic']['predictions'][-1]

print(f"""
📌 FINAL FORECAST OF HOUSEHOLD DEPOSIT VOLUME IN RUSSIA
   Period: March 2026 — February 2027

1. CURRENT LEVEL (February 2026): {last_depos:,.0f} billion RUB

2. BASELINE SCENARIO:
   By February 2027: {base_final:,.0f} billion RUB
   Growth over 12 months: {(base_final - last_depos) / last_depos * 100:+.1f}%
   95% CI: [{lower_bound[-1]:,.0f} — {upper_bound[-1]:,.0f}] billion RUB

3. OPTIMISTIC SCENARIO:
   By February 2027: {opt_final:,.0f} billion RUB
   Growth over 12 months: {(opt_final - last_depos) / last_depos * 100:+.1f}%

4. PESSIMISTIC SCENARIO:
   By February 2027: {pes_final:,.0f} billion RUB
   Growth over 12 months: {(pes_final - last_depos) / last_depos * 100:+.1f}%

5. KEY DRIVERS:
   - WAGE (salary): main macroeconomic factor
   - DEPOS_lag_1: deposit inertia
   - post_2022: structural shift after 2022

6. METHODOLOGY:
   - Model: Ridge regression (alpha=1.0), 37 features
   - Trained on all 134 observations (Jan 2015 — Feb 2026)
   - Structure identical to validated model (R²_test = 0.9422)
   - Iterative forecast with DEPOS lag updates

7. LIMITATIONS:
   - Forecast based on simplified macro factor scenarios
   - Does not account for possible shocks (crises, sanctions)
   - Accuracy decreases with forecast horizon
   - Model on 134 observations lacks independent validation

8. RECOMMENDATIONS:
   - Use baseline scenario for planning
   - Track actual WAGE, CPI, UNEM values
   - Update forecast monthly as new data becomes available
""")

print("✅ BLOCK 6 COMPLETED")
print("📌 PROJECT FULLY COMPLETED")